In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough, RunnableParallel 
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

In [3]:
import os

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [12]:
questions = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em \"Os Sertões\"? Como esses aspectos refletem a visão do autor sobre o Brasil da época?",
]

In [5]:
## Embedding | LLM

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=300)

In [6]:
# PDF

pdf_link = "../os-sertoes.pdf"
loader = PyPDFLoader(pdf_link, extract_images=False)
pages = loader.load_and_split()

In [7]:
# Chunking

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000, 
    chunk_overlap=20,
    length_function=len,
    add_start_index=True
)

chunks = text_splitter.split_documents(pages)

In [8]:
# Save chunks 
vectordb = Chroma(embedding_function=embedding_model, persist_directory="rerankDB")

/tmp/ipykernel_205287/1575809056.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(embedding_function=embedding_model, persist_directory="rerankDB")


In [9]:
# Load db

naive_retriever = vectordb.as_retriever(search_kwargs={"k": 10})


In [ ]:
os.environ["COHERE_API_KEY"] = ""

In [11]:
rerank = CohereRerank(model="rerank-multilingua-v3.0", top_n=3)

compressor_retriever = ContextualCompressionRetriever(
    base_compressor=rerank,
    base_retriever=naive_retriever
)

In [13]:
TEMPLATE = """
    Você é um assistente de perguntas e respostas sobre o livro "Os Sertões" de Euclides da Cunha. Responda a pergunta abaixo utilizando o contexto informado.
    Context: {context}
    Pergunta: {question}
"""

prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [14]:
setup_retrieval = RunnableParallel({"question": RunnablePassthrough(), "context": compressor_retriever})
output_parser = StrOutputParser()
compressor_retrieval_chain = setup_retrieval | prompt | llm | output_parser

In [15]:
for index, question in enumerate(questions):
    result = {"output": compressor_retrieval_chain.invoke(question)}
    answer = result["output"]
    print({"numero": index+1, "pergunta": question, "resposta": answer})



{'numero': 1, 'pergunta': 'Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?', 'resposta': 'Euclides da Cunha descreve o ambiente natural do sertão nordestino como árido, inóspito e desafiador, caracterizado pela seca, sol escaldante, vegetação espinhosa e escassez de recursos hídricos. Ele acredita que esse ambiente hostil exerce uma forte influência na vida dos habitantes locais, moldando sua cultura, modo de vida e até mesmo sua psique. Os sertanejos são retratados como povo resiliente, adaptado às condições adversas do sertão, porém também marcados pela luta constante pela sobrevivência e pela fatalidade da seca, que muitas vezes leva à miséria e à violência. Euclides da Cunha destaca a relação intrínseca entre o homem e o meio ambiente no sertão nordestino, enfatizando como o cenário árido e impiedoso molda a vida e a identidade dos habitantes da região.'}
{'numero': 2, 'pergunta': 'Quais são as principai